In [1]:
# ============================================================
# CELL 1 - Install Hugging Face Hub
# ============================================================

!pip install huggingface_hub -q
print("huggingface_hub installed!")

huggingface_hub installed!


In [2]:
# ============================================================
# CELL 2 - Login to Hugging Face
# ============================================================

from huggingface_hub import login
from getpass import getpass

# Get token from https://huggingface.co/settings/tokens
HF_TOKEN = getpass("Enter Hugging Face token: ")
login(token=HF_TOKEN)
print("Logged in to Hugging Face!")

Enter Hugging Face token: ··········
Logged in to Hugging Face!


In [3]:
# ============================================================
# CELL 3 - Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Verify checkpoints are available
import os, glob

drive_base = "/content/drive/MyDrive/advertising_panel_segmentation/results"

print("=== Available checkpoints on Drive ===")
for exp in ['exp0_segformer_b0_baseline',
            'exp1_segformer_b1_standard',
            'exp2_segformer_b1_augmented',
            'exp3_segformer_b1_optimized',
            'exp3_segformer_b1_optimized2',
            'exp3_segformer_b1_optimized3']:
    path = f"{drive_base}/{exp}"
    pth_files = glob.glob(f"{path}/best_mIoU_iter_*.pth")
    if pth_files:
        best = max(pth_files, key=lambda x: int(x.split('iter_')[1].split('.')[0]))
        size = os.path.getsize(best) / (1024**2)
        print(f"  ✅ {exp}: {os.path.basename(best)} ({size:.1f} MB)")
    else:
        print(f"  ❌ {exp}: no checkpoint found")

# Verify dataset
print("\n=== Dataset on Drive ===")
dataset_path = "/content/drive/MyDrive/Dataset/processed.zip"
if os.path.exists(dataset_path):
    size = os.path.getsize(dataset_path) / (1024**2)
    print(f"  ✅ processed.zip ({size:.1f} MB)")
else:
    print(f"  ❌ processed.zip not found at {dataset_path}")

Mounted at /content/drive
=== Available checkpoints on Drive ===
  ✅ exp0_segformer_b0_baseline: best_mIoU_iter_18000.pth (16.5 MB)
  ✅ exp1_segformer_b1_standard: best_mIoU_iter_10000.pth (53.5 MB)
  ✅ exp2_segformer_b1_augmented: best_mIoU_iter_14000.pth (54.0 MB)
  ✅ exp3_segformer_b1_optimized: best_mIoU_iter_14000.pth (58.1 MB)
  ✅ exp3_segformer_b1_optimized2: best_mIoU_iter_12000.pth (53.8 MB)
  ✅ exp3_segformer_b1_optimized3: best_mIoU_iter_18000.pth (54.5 MB)

=== Dataset on Drive ===
  ✅ processed.zip (386.7 MB)


In [4]:
# ============================================================
# CELL 4 - Create Hugging Face Repository
# ============================================================

from huggingface_hub import HfApi

api = HfApi()

REPO_ID = "ilMassy/advertising-panel-segmentation"  # username/repo-name

# Create repository (type=model to support both models and datasets)
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,  # no error if already exists
)

print(f"Repository created: https://huggingface.co/{REPO_ID}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Repository created: https://huggingface.co/ilMassy/advertising-panel-segmentation


In [9]:
# ============================================================
# CELL 5 - Create Repository README
# ============================================================

readme_content = """---
license: mit
tags:
  - segmentation
  - semantic-segmentation
  - segformer
  - sports
  - advertising-boards
  - mmsegmentation
---

# Advertising Panel Segmentation — SegFormer

Semantic segmentation models for detecting advertising LED boards in sports broadcasting footage.

## Dataset
- **967 images** at 1920x1080 from sports highlights
- **Manually annotated** via CVAT.ai with polygon annotations
- **Binary segmentation**: background (0) vs advertising board (1)
- **Split**: Train 672 / Val 149 / Test 146

## Models

| Experiment | Model | Augmentation | mIoU | Board IoU | Dice | Precision | Recall |
|---|---|---|---|---|---|---|---|
| Exp0 | SegFormer-B0 | Standard | 87.15% | 76.28% | 86.55% | 84.13% | 89.10% |
| Exp1 | SegFormer-B1 | Standard | 84.29% | 71.12% | 83.12% | 79.26% | 87.39% |
| **Exp2** | **SegFormer-B1** | **Sport-specific** | **87.26%** | **76.45%** | **86.66%** | **85.76%** | **87.57%** |
| Exp3-opt1 | SegFormer-B1 | Reduced + LR 3e-5 + channels=512 | 84.58% | 71.68% | 83.51% | 78.66% | 88.99% |
| Exp3-opt2 | SegFormer-B1 | Reduced + LR 3e-5 + channels=256 | 84.92% | 72.26% | 83.90% | 80.40% | 87.71% |
| Exp3-opt3 | SegFormer-B1 | Standard + drop_path=0.15 | 86.42% | 74.99% | 85.71% | 82.25% | 89.47% |

Best model: **Exp2 - SegFormer-B1 Augmented** (`models/exp2_segformer_b1_augmented/best_mIoU_iter_14000.pth`)

## Detailed Results

### Exp0 - SegFormer-B0 Baseline
- Best checkpoint: best_mIoU_iter_18000.pth
- mIoU: 87.15% | Board IoU: 76.28% | Dice: 86.55% | Precision: 84.13% | Recall: 89.10%

### Exp1 - SegFormer-B1 Standard
- Best checkpoint: best_mIoU_iter_10000.pth
- mIoU: 84.29% | Board IoU: 71.12% | Dice: 83.12% | Precision: 79.26% | Recall: 87.39%

### Exp2 - SegFormer-B1 Augmented ⭐ Best
- Best checkpoint: best_mIoU_iter_14000.pth
- mIoU: 87.26% | Board IoU: 76.45% | Dice: 86.66% | Precision: 85.76% | Recall: 87.57%

### Exp3 - SegFormer-B1 Optimized1
- Best checkpoint: best_mIoU_iter_14000.pth
- mIoU: 84.58% | Board IoU: 71.68% | Dice: 83.51% | Precision: 78.66% | Recall: 88.99%

### Exp3 - SegFormer-B1 Optimized2
- Best checkpoint: best_mIoU_iter_12000.pth
- mIoU: 84.92% | Board IoU: 72.26% | Dice: 83.90% | Precision: 80.40% | Recall: 87.71%

### Exp3 - SegFormer-B1 Optimized3
- Best checkpoint: best_mIoU_iter_18000.pth
- mIoU: 86.42% | Board IoU: 74.99% | Dice: 85.71% | Precision: 82.25% | Recall: 89.47%

## Framework
- PyTorch 2.4.1
- MMSegmentation 1.2.2
- mmcv 2.2.0

## Repository Structure
```
models/
  exp0_segformer_b0_baseline/best_mIoU_iter_18000.pth
  exp1_segformer_b1_standard/best_mIoU_iter_10000.pth
  exp2_segformer_b1_augmented/best_mIoU_iter_14000.pth
  exp3_segformer_b1_optimized/best_mIoU_iter_14000.pth
  exp3_segformer_b1_optimized2/best_mIoU_iter_12000.pth
  exp3_segformer_b1_optimized3/best_mIoU_iter_18000.pth
dataset/
  processed.zip  (967 images + masks, train/val/test split)
```

## GitHub
https://github.com/ilMassy/advertising-panel-segmentation
"""

with open('/content/README.md', 'w') as f:
    f.write(readme_content)

api.upload_file(
    path_or_fileobj='/content/README.md',
    path_in_repo='README.md',
    repo_id=REPO_ID,
    repo_type="model",
)
print("README uploaded!")

README uploaded!


In [6]:
# ============================================================
# CELL 6 - Upload Model Checkpoints
# ============================================================

from huggingface_hub import HfApi
import glob, os

api = HfApi()

experiments = [
    'exp0_segformer_b0_baseline',
    'exp1_segformer_b1_standard',
    'exp2_segformer_b1_augmented',
    'exp3_segformer_b1_optimized',
    'exp3_segformer_b1_optimized2',
    'exp3_segformer_b1_optimized3',
]

drive_base = "/content/drive/MyDrive/advertising_panel_segmentation/results"

for exp in experiments:
    pth_files = glob.glob(f"{drive_base}/{exp}/best_mIoU_iter_*.pth")

    if not pth_files:
        print(f"❌ {exp}: no checkpoint found, skipping")
        continue

    best = max(pth_files, key=lambda x: int(x.split('iter_')[1].split('.')[0]))
    filename = os.path.basename(best)
    size = os.path.getsize(best) / (1024**2)

    print(f"Uploading {exp}/{filename} ({size:.1f} MB)...")

    api.upload_file(
        path_or_fileobj=best,
        path_in_repo=f"models/{exp}/{filename}",
        repo_id=REPO_ID,
        repo_type="model",
    )
    print(f"  ✅ Uploaded!")

print("\nAll checkpoints uploaded!")

Uploading exp0_segformer_b0_baseline/best_mIoU_iter_18000.pth (16.5 MB)...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../best_mIoU_iter_18000.pth:   4%|3         |  607kB / 17.3MB            

  ✅ Uploaded!
Uploading exp1_segformer_b1_standard/best_mIoU_iter_10000.pth (53.5 MB)...


...

  ✅ Uploaded!
Uploading exp3_segformer_b1_optimized3/best_mIoU_iter_18000.pth (54.5 MB)...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../best_mIoU_iter_18000.pth:   0%|          |  129kB / 57.2MB            

  ✅ Uploaded!

All checkpoints uploaded!


In [7]:
# ============================================================
# CELL 7 - Upload Dataset
# ============================================================

from huggingface_hub import HfApi
import os

api = HfApi()

dataset_path = "/content/drive/MyDrive/Dataset/processed.zip"
size = os.path.getsize(dataset_path) / (1024**2)

print(f"Uploading dataset ({size:.1f} MB)...")

api.upload_file(
    path_or_fileobj=dataset_path,
    path_in_repo="dataset/processed.zip",
    repo_id=REPO_ID,
    repo_type="model",
)

print("✅ Dataset uploaded!")
print(f"\nRepository available at: https://huggingface.co/{REPO_ID}")

Uploading dataset (386.7 MB)...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ive/Dataset/processed.zip:   1%|          | 3.67MB /  405MB            

✅ Dataset uploaded!

Repository available at: https://huggingface.co/ilMassy/advertising-panel-segmentation


In [8]:
# ============================================================
# CELL 8 - Verify Upload
# ============================================================

from huggingface_hub import HfApi

api = HfApi()

print(f"=== Files in {REPO_ID} ===")
files = api.list_repo_files(repo_id=REPO_ID, repo_type="model")
for f in files:
    print(f"  {f}")

=== Files in ilMassy/advertising-panel-segmentation ===
  .gitattributes
  README.md
  dataset/processed.zip
  models/exp0_segformer_b0_baseline/best_mIoU_iter_18000.pth
  models/exp1_segformer_b1_standard/best_mIoU_iter_10000.pth
  models/exp2_segformer_b1_augmented/best_mIoU_iter_14000.pth
  models/exp3_segformer_b1_optimized/best_mIoU_iter_14000.pth
  models/exp3_segformer_b1_optimized2/best_mIoU_iter_12000.pth
  models/exp3_segformer_b1_optimized3/best_mIoU_iter_18000.pth
